# PISA reproduction workflow (local-only)

This notebook runs on deterministic synthetic data by default. Set `PISA_INPUT` and `PISA_MAPPING` to reviewed local paths for a PISA run. Raw rows are never written by this notebook.

## Goal and boundary

Fit the interpretable binary pairwise energy model and compare aggregate moments. Ordinary row weights are supported; replicate weights, plausible-value aggregation, and complex-survey variance are outside this package.

In [1]:
import json
import os
from pathlib import Path

import numpy as np

from learning_energy_model import (
    DataConfig,
    LearningModel,
    PISAMapping,
    compare_moment_orders,
    prepare_pisa_file,
)

In [2]:
input_path = os.getenv("PISA_INPUT")
mapping_path = os.getenv("PISA_MAPPING")
if bool(input_path) != bool(mapping_path):
    raise ValueError("set both PISA_INPUT and PISA_MAPPING, or neither")

if input_path:
    mapping = PISAMapping.load(mapping_path)
    prepared = prepare_pisa_file(input_path, mapping)
    feature_names, target_name = prepared.feature_names, prepared.target_name
    X, y, sample_weight = prepared.X, prepared.y, prepared.sample_weight
    source_mode = "local_pisa"
else:
    rng = np.random.default_rng(7)
    X = rng.integers(0, 2, size=(400, 2)).astype(float)
    y = ((X[:, 0] + X[:, 1] + rng.integers(0, 2, size=400)) >= 2).astype(float)
    feature_names, target_name = ("feature_a", "feature_b"), "target"
    sample_weight, source_mode = None, "synthetic_smoke"
print({"source_mode": source_mode, "rows": len(y), "features": list(feature_names)})

{'source_mode': 'synthetic_smoke', 'rows': 400, 'features': ['feature_a', 'feature_b']}


### Fit and inspect aggregate moments

In [3]:
model = LearningModel(DataConfig(feature_names=tuple(feature_names), target_name=target_name, missing_strategy="median"), calculation="auto", seed=7)
fit = model.fit(X, y, sample_weight=sample_weight)
analysis = model.analyze()
comparison = compare_moment_orders(
    {1: fit.observed_means, 2: fit.observed_pairwise_moments, 3: fit.observed_higher_order_moments},
    {1: fit.model_means, 2: fit.model_pairwise_moments, 3: fit.model_higher_order_moments},
    tolerances={1: 0.08, 2: 0.08, 3: 0.08},
)
report = {"source_mode": source_mode, "nodes": [*feature_names, target_name], "calculation": fit.diagnostics, "fit": fit.to_dict(), "moment_comparison": comparison.to_dict(), "parameters": {"h": analysis.h.tolist(), "J": analysis.J.tolist()}}
print(json.dumps({"calculation": report["calculation"], "moment_comparison": report["moment_comparison"]}, indent=2, ensure_ascii=False))

{
  "calculation": {
    "n_samples": 400,
    "n_nodes": 3,
    "calculation": "exact",
    "training_method": "moment_matching",
    "quality_passed": true,
    "quality_checks": {}
  },
  "moment_comparison": {
    "max_absolute_error": {
      "1": 0.010902786678804177,
      "2": 0.010902786678804177,
      "3": 0.015495121426269798
    },
    "mean_absolute_error": {
      "1": 0.0036897861908983254,
      "2": 0.00754845593139523,
      "3": 0.015495121426269798
    },
    "tolerances": {
      "1": 0.08,
      "2": 0.08,
      "3": 0.08
    },
    "passed": true
  }
}


In [4]:
assert report["moment_comparison"]["passed"]
assert np.isfinite(analysis.h).all() and np.isfinite(analysis.J).all()
output_path = os.getenv("PISA_REPORT_OUTPUT")
if output_path:
    Path(output_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
print("aggregate checks passed; raw observations were not exported")

aggregate checks passed; raw observations were not exported


## Next steps

For a real run, validate the mapping JSON against the matching OECD codebook, set `PISA_INPUT`, `PISA_MAPPING`, and optionally `PISA_REPORT_OUTPUT`, then rerun top-to-bottom. Interpret results as prepared-sample model estimates, not official PISA estimates.